In [0]:
from pyspark.sql.functions import col, count, countDistinct, min, max, avg, when

CATALOG = "czech_fintech"
SILVER = "silver"

print("=" * 80)
print("SILVER LAYER QUALITY CHECKS")
print("=" * 80)

# ============================================================================
# TRANSACTIONS
# ============================================================================
print("\n1️⃣ TRANSACTIONS")
trans = spark.table(f"{CATALOG}.{SILVER}.transactions")

print(f"Total rows: {trans.count()}")
print(f"Distinct trans_id: {trans.select('trans_id').distinct().count()}")
print(f"Date range: {trans.select(min('date'), max('date')).show()}")
print(f"Amount range (min/max): {trans.select(min('amount'), max('amount')).show()}")
print(f"NULL check:")
trans.select(
    countDistinct(when(col("trans_id").isNull(), 1)).alias("trans_id_nulls"),
    countDistinct(when(col("account_id").isNull(), 1)).alias("account_id_nulls"),
    countDistinct(when(col("client_id").isNull(), 1)).alias("client_id_nulls"),
    countDistinct(when(col("amount").isNull(), 1)).alias("amount_nulls"),
).show()

# ============================================================================
# ACCOUNT
# ============================================================================
print("\n2️⃣ ACCOUNT")
account = spark.table(f"{CATALOG}.{SILVER}.account")

print(f"Total accounts: {account.count()}")
print(f"Distinct district_id: {account.select('district_id').distinct().count()}")
print(f"Frequency values:")
account.select("frequency").distinct().show()

# ============================================================================
# CLIENT
# ============================================================================
print("\n3️⃣ CLIENT")
client = spark.table(f"{CATALOG}.{SILVER}.client")

print(f"Total clients: {client.count()}")
print(f"Birth date range:")
client.select(min("birth_number"), max("birth_number")).show()

# ============================================================================
# DISP
# ============================================================================
print("\n4️⃣ DISP (account-client link)")
disp = spark.table(f"{CATALOG}.{SILVER}.disp")

print(f"Total dispositions: {disp.count()}")
print(f"Disposition types:")
disp.select("type").distinct().show()
print(f"Accounts per client:")
disp.groupBy("client_id").agg(countDistinct("account_id").alias("account_count")).agg(
    min("account_count"), max("account_count"), avg("account_count")
).show()

# ============================================================================
# JOIN COVERAGE CHECK
# ============================================================================
print("\n5️⃣ JOIN COVERAGE")

# Transactions vs Account coverage
trans_acct_coverage = trans.join(
    account.select("account_id"),
    "account_id",
    "inner"
).count()

print(f"Transactions matched with account: {trans_acct_coverage} / {trans.count()}")

# Transactions vs Client coverage (through disp)
trans_client_coverage = trans.select("client_id").filter(col("client_id").isNotNull()).count()
print(f"Transactions with client_id: {trans_client_coverage} / {trans.count()}")

# ============================================================================
# LOAN
# ============================================================================
print("\n6️⃣ LOAN")
loan = spark.table(f"{CATALOG}.{SILVER}.loan")

print(f"Total loans: {loan.count()}")
print(f"Loan status values:")
loan.select("status").distinct().show()
print(f"Amount range:")
loan.select(min("amount"), max("amount")).show()

# ============================================================================
# ORDER
# ============================================================================
print("\n7️⃣ ORDER")
order = spark.table(f"{CATALOG}.{SILVER}.order")

print(f"Total orders: {order.count()}")
print(f"Amount range:")
order.select(min("amount"), max("amount")).show()

# ============================================================================
# CARD
# ============================================================================
print("\n8️⃣ CARD")
card = spark.table(f"{CATALOG}.{SILVER}.card")

print(f"Total cards: {card.count()}")
print(f"Card types:")
card.select("type").distinct().show()

# ============================================================================
# DISTRICT
# ============================================================================
print("\n9️⃣ DISTRICT")
district = spark.table(f"{CATALOG}.{SILVER}.district")

print(f"Total districts: {district.count()}")
print(f"Sample districts:")
district.select("A1", "A2").limit(5).show()

print("\n" + "=" * 80)
print("✅ SILVER QUALITY CHECKS COMPLETE")
print("=" * 80)